# 7 · Stats  `[EVAL]`

The **full statistical tables** behind the figures in `1`–`4` — kept here so the analysis notebooks stay figure-led. Everything is full-conversation eval, paired by the 96 shared personas; thin arms (<3 scored iters) are dropped to avoid NaN rows. Exports → `results/<VIEW>/tables/7_stats/`. For a reader who wants exact numbers.

## 0 · Confirmatory vs exploratory — read this first  `[EVAL]`

**Multiplicity scope.** Every *p*-value below is Holm- or BH-corrected **within** its own family (across rubrics within one matched *(K/method, iteration)* contrast, or across iterations within one arm-vs-base sweep), but corrections are **not** pooled across the dozens of families in this EDA. To keep the inference honest, the analyses split into a small pre-registered **confirmatory** set (the thesis claims) and a larger **exploratory** set (hypothesis-generating — report descriptively; don't over-read an isolated star).

**Confirmatory (primary hypotheses).**
- **PTO > GRPO on Q1+Q2 at matched budget** — reported at **both** the matched *final* iteration (§4) **and** as the **best-vs-best model-selection contrast** (§4b, `method_paired_best`: PTO at its own-oracle best iteration vs GRPO at its iter-8 peak), so GRPO is credited at its peak (≈4.08) rather than only at its regressed endpoint. PTO wins on both (best-vs-best +0.18 on Q1+Q2; final 4.26 vs 3.75).
- **Each arm improves over its own base** on the primary metric (Q1+Q2), at both final and best iteration (§1, `target` column).
- **The reward-hacking signature on the metrics the reward never saw** — MICI (MI-inconsistency) rises and patient change-talk / MI technique fail to keep pace as the global-eval (halo) scores climb (§4 here; the synthesis in `3_Validity_and_Hacking` §2).

**Exploratory (hypothesis-generating).** Per-subscale/item trajectories (`2_Questionnaire_Detail`); per-persona-trait subgroups (`4_Heterogeneity`); the GRPO iter-9 anomaly check (§6); the full per-iteration vs-base sweeps (§3); **RQ-i, the K0-vs-K5 look-ahead contrast (§4c)** — now a genuine K×method comparison (PTO K=5 to iter 10, GRPO K=5 to iter 5), with the reward-faithfulness side of the same question in `5_Training` §4 and the **cost** side in §4e. ⚠ Read §4c against §4e: the matched-*iteration* contrast is not budget-matched, and on `MICI` the two axes disagree in sign.

In [ ]:
import sys, os
_p = os.path.abspath(".")                      # find eda/ (the dir holding eda_analysis/) from any depth
while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "eda_analysis")):
    _p = os.path.dirname(_p)
sys.path.insert(0, _p)
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
pd.set_option("display.width", 185, "display.max_columns", 50)
import eda_analysis
from eda_analysis import stats

# ╔═══ VIEW — the one knob ════════════════════════════════════════════════════════╗
# "all" = every arm | "L0" = K=0 arms (PTO_LA0/GRPO_LA0) | "L5" = K=5 arms (thin, paused).
# Sets BOTH the arm filter AND the results root -> results/<VIEW>/figures|tables/<group>/.
# Edit the default for interactive use; render_views.py overrides it via the EDA_VIEW env var.
VIEW = os.environ.get("EDA_VIEW", "L0")
# ╔═══ JUDGE — which grader's scores to read ══════════════════════════════════════╗
# "" = the primary oracle (gpt-4o-mini) eval_scores/ tree, i.e. the numbers the thesis reports.
# A judge tag (e.g. "anthropic_claude-haiku-4-5") reads that judge's partition of the score lake instead and
# routes exports to results/judges/<tag>/<VIEW>/. Orthogonal to VIEW: VIEW filters ARMS, JUDGE
# selects the SCORE SOURCE. render_views.py overrides it via the EDA_JUDGE env var.
JUDGE = os.environ.get("EDA_JUDGE", "")

cfg = eda_analysis.EdaConfig(
    view=VIEW, judge=JUDGE,                             # arm filter + results/<VIEW>/ root
    export_group="7_stats",                # topic family = this notebook's number
    selection="all",
    focus_arms=None, focus_metric="Q1Q2",
)
S = eda_analysis.notebook_setup(cfg)
FOCUS = cfg.focus_arms or sorted(S.SCORES.arm.unique())

## 1 · Main results — each arm vs its own base  `[EVAL]`
**Purpose.** Per (arm × metric): the FINAL and the BEST iteration vs base in one table (column `target`) — Δ, paired Cohen's *dz* + label, Wilcoxon *p* (Holm), bootstrap 95% CI, trajectory ρ / slope.

In [ ]:
MR  = stats.filter_thin_arms(stats.main_results_table(S.SCORES, target="final"), S.SCORES)
MRb = stats.filter_thin_arms(stats.main_results_table(S.SCORES, target="best"),  S.SCORES)
MRall = pd.concat([MR.assign(target="final"), MRb.assign(target="best")], ignore_index=True)
MRall = MRall[["arm", "rubric", "target", "base", "target_iter"] +
              [c for c in MRall.columns if c not in ("arm", "rubric", "target", "base", "target_iter")]]
display(MRall)
eda_analysis.save_table(MRall, "main_results", caption="FINAL and BEST iteration vs base per arm x metric (column `target`), persona-paired (N=96): dz, Wilcoxon p (Holm within arm x target), bootstrap CI, trajectory rho/slope. Thin arms dropped.")

## 2 · Repeated-measures omnibus (Friedman)  `[EVAL]`
**Purpose.** Is iteration a real within-persona factor? Friedman χ² + Kendall's *W* per arm × rubric.

In [ ]:
FR = pd.DataFrame([stats.friedman_trajectory(S.SCORES, a, m)
                   for a in sorted(S.SCORES.arm.unique()) for m in S.METRICS])
FR = stats.filter_thin_arms(FR, S.SCORES)
display(FR.round(4))
eda_analysis.save_table(FR.round(4), "friedman_omnibus", caption="Friedman repeated-measures omnibus across iterations per arm x rubric (Kendall's W). Thin arms dropped.")

## 3 · Per-arm vs-base, every iteration (paired)  `[EVAL]`
**Purpose.** The full iteration-by-iteration Q1+Q2 vs-base table per arm (each arm vs its OWN base).

In [ ]:
frames = []
for arm in sorted(S.SCORES.arm.unique()):
    if arm in stats.thin_arms(S.SCORES): continue
    PV = stats.paired_vs_base(S.SCORES, arm, "Q1Q2")
    if not PV.empty: frames.append(PV)
if frames:
    VB = pd.concat(frames, ignore_index=True)[["arm", "iteration", "n", "mean_delta", "dz", "p", "p_holm", "ci_low", "ci_high"]].round(4)
    display(VB)
    eda_analysis.save_table(VB, "vs_base_paired", caption="Each arm x iteration vs its OWN base on Q1+Q2 — one merged table (column `arm`); persona-paired Wilcoxon, dz, Holm p (within arm), bootstrap CI.")
else:
    print("no non-thin arms scored.")

## 4 · Cross-arm contrasts — PTO vs GRPO at matched K  `[EVAL]`
**Purpose.** The method contrast as a paired table at matched iterations (iteration-0 base rows dropped). Positive `mean_delta` = first term higher. **Holm scope:** each row's `p_holm` is corrected across the rubrics *within its own (K, iteration) contrast* — every matched-budget point is its own family, so `p_holm` is **not** pooled across iterations. §4b adds the **best-vs-best** model-selection contrast (different iterations per side — the honest "each method's selected checkpoint" comparison); §4c holds the **look-ahead** contrast (K0 vs K5), which needs both K arms and so is owned by a single view.

In [ ]:
frames = []
for K in sorted(S.SCORES.K.unique()):
    CMP = stats.paired_method_comparison(S.SCORES, "PTO", "GRPO", K=K)
    if not CMP.empty: frames.append(CMP[CMP.iteration > 0])
if frames:
    MP = pd.concat(frames, ignore_index=True)
    view = MP[["K", "iteration", "metric", "n", "mean_delta", "dz", "p_holm"]].round(4)
    print("=== PTO - GRPO at matched K + iterations (+ => PTO higher) ==="); display(view)
    eda_analysis.save_table(view, "method_paired_by_K", caption="PTO - GRPO at matched K and matched iterations — one merged table (column `K`); persona-paired Wilcoxon + dz + Holm. + => PTO higher. NOTE Holm scope: p_holm is corrected across the rubrics WITHIN each (K, iteration) contrast, NOT across iterations (each matched-budget point is its own family).")
else:
    print("no common PTO/GRPO iterations at any K.")

### 4b · Best-vs-best — the model-selection contrast  `[EVAL]`
**Purpose.** PTO at its **own-oracle best** iteration vs GRPO at **its** best (per `best_per_experiment`; e.g. PTO@10 vs GRPO@8) — persona-paired across the different iterations (valid: every iteration reshuffles the same 96 personas). This is the strongest-steelman comparison: GRPO is credited at its peak, before the post-peak regression. Complements the matched-iteration table above; on thin-K rows it is **descriptive only**.

In [ ]:
frames = []
for K in sorted(S.SCORES.K.unique()):
    BB = stats.paired_best_method_comparison(S.SCORES, "PTO", "GRPO", K=K)
    if not BB.empty: frames.append(BB)
if frames:
    BBall = pd.concat(frames, ignore_index=True)
    view = BBall[["K", "iter_a", "iter_b", "metric", "n", "mean_delta", "dz", "p_holm"]].round(4)
    print("=== PTO(best) - GRPO(best) per K (+ => PTO higher; iter_a=PTO best, iter_b=GRPO best) ===")
    display(view)
    eda_analysis.save_table(view, "method_paired_best", caption="PTO at its own-oracle BEST iteration vs GRPO at ITS best, per K — persona-paired Wilcoxon + dz + Holm (across rubrics within each K). + => PTO higher. The model-selection contrast: GRPO credited at its peak (iter_b), before the post-peak regression; complements the matched-endpoint table method_paired_by_K. Thin-K rows are descriptive only.")
else:
    print("no K with both PTO and GRPO best models scored.")

### 4c · RQ-i — does look-ahead help? K=0 vs K=5 within each method  `[EVAL]`
**Purpose.** The look-ahead lever, isolated: the same method at K=0 vs K=5, persona-paired at every iteration both arms reached. Three artifacts — `k_means_by_iter` (where the two arms sit, per iteration), `k_paired_by_method` (the paired Δ / *dz* / Holm *p*), and `k_trajectory_Q1Q2` (both K arms of both methods in one frame).

**Why this section is special.** Every other contrast here reads `S.SCORES`, which the VIEW has already filtered to ONE K — so `L0` sees only the K=0 arms and `L5` only the K=5 arms, and a K contrast is empty in both. This block instead reads `eda_analysis.cross_k_scores(S)`, which rebuilds the score frame with the K filter (and only the K filter) dropped. To keep one owner per fact, the artifacts are saved **only in the `RQ_I_VIEW`** (`L5`, whose SUMMARY narrates RQ-i); the other views print a pointer. Export routing is untouched — `cross_k_scores` changes what is read, never where things are written.

**Read it against §4e.** Both K=5 arms are now trained well past a prefix (PTO to iter 10, GRPO to iter 5), so the matched-iteration comparison is real — but it is matched on *iterations*, and a K=5 iteration costs ~1.9× a K=0 one. §4e re-runs the same question at matched GPU-hours, where the two GRPO arms turn out to be budget-matched to within ~3% and the `MICI` contrast reverses sign. Quote both, or say which axis you mean. Sign convention: **+ ⇒ K=0 higher** (i.e. a positive Δ means look-ahead *cost* score).

In [ ]:
if S.VIEW != eda_analysis.RQ_I_VIEW:
    print(f"[skip] RQ-i (K0 vs K5) is owned by the {eda_analysis.RQ_I_VIEW} view — see "
          f"results/{eda_analysis.RQ_I_VIEW}/tables/7_stats/k_{{means_by_iter,paired_by_method}} "
          f"+ figures/7_stats/k_trajectory_{cfg.focus_metric}.")
else:
    KS = eda_analysis.cross_k_scores(S)      # the K filter dropped; exports still land in THIS view
    print("cross-K frame:", KS.shape, "| arms:", sorted(KS.arm.unique()))

    KM = pd.concat([M for M in (stats.k_means_by_iter(KS, m) for m in ["PTO", "GRPO"]) if not M.empty],
                   ignore_index=True).round(4)
    print("=== K=0 vs K=5 arm MEANS per iteration (levels; + delta => K0 higher) ==="); display(KM)
    eda_analysis.save_table(KM, "k_means_by_iter", caption="RQ-i LEVELS: K=0 vs K=5 arm means per method x rubric x iteration, with the unpaired delta (+ => K0 higher) and each side's n. Iterations the K=5 arm never reached keep mean_K5/delta NaN (n_K5=0) — K=0 continuing past the K=5 arm's last iteration is part of the look-ahead answer. Read dz/p off k_paired_by_method, NOT here: delta is a difference of arm means (equal to the paired mean_delta when both cells are complete, but carrying none of the pairing's precision). Built from cross_k_scores, the only frame in which either K-specific view holds both arms.")

    frames = []
    for method in ["PTO", "GRPO"]:
        CMP = stats.paired_k_comparison(KS, method)
        if not CMP.empty: frames.append(CMP[CMP.iteration > 0])
    if frames:
        KP = pd.concat(frames, ignore_index=True)
        view = KP[["method", "iteration", "metric", "n", "mean_delta", "dz", "p", "p_holm"]].round(4)
        print("=== K0 - K5 within each method, persona-paired (+ => K0 higher) ==="); display(view)
        eda_analysis.save_table(view, "k_paired_by_method", caption="RQ-i TEST: K0 - K5 within each method at matched iterations — one merged table (column `method`); persona-paired Wilcoxon + dz + Holm. + => K0 higher, so a positive delta means look-ahead COST score. NOTE Holm scope: p_holm is corrected across rubrics WITHIN each (method, iteration) contrast, not across iterations. The K=5 arms are thin (PTO to iter 5, GRPO to iter 1) — matched-iteration and descriptive, not an endpoint claim.")
    else:
        print("K0-vs-K5 not comparable yet for either method.")

    fig = eda_analysis.plotting.single_metric_trajectory(
        KS, cfg.focus_metric, palette=eda_analysis.plotting.arm_palette(sorted(KS.arm.unique())),
        oracle_noise=S.ORACLE_NOISE)
    eda_analysis.save_fig(fig, f"k_trajectory_{cfg.focus_metric}", caption="RQ-i in one frame: both look-ahead arms of both methods overlaid on the primary metric (mean +/- 95% CI over the 96 personas). The K=5 lines STOP at their last scored iteration (PTO 10, GRPO 5) - everything to the right is K=0 only, so compare the arms in the overlapping region and read the tail as 'K=0 kept training', not as a K difference. For GRPO that tail is NOT free: see the compute axis in 4e, where GRPO's two arms cost the same total GPU-hours.")
    plt.show()

### 4d · RQ-i on the BEHAVIOUR channels — what the reward curve hides  `[EVAL]`
**Purpose.** §4c asks the K question of the eight *rubrics*. This asks it of the *behaviours those rubrics are made of* — the MI-inconsistent acts, the MITI-coded acts, and the deterministic session shape — through the same persona-paired machinery, from `behavior.channel_scores_long` (shaped exactly like `scores_long`, so nothing new is computed here: it is a different frame fed to `stats.paired_k_comparison`).

**Why it needs its own section.** A rubric is a weighted summary of behaviours, and a summary can be flat while its components move in opposite directions. That is not hypothetical here: on `Q1+Q2` the two K arms are within *dz* 0.17 at the matched endpoint, and on `MICI_OverPraise_rate` they are a *large* effect apart in the same conversations.

**⚠ Read the per-SESSION family before believing anything in the per-TURN family.** Two controls have to pass before a rate gap is a behaviour claim, and on this data one of them fails in an informative way:

1. **Denominator, and it points in OPPOSITE directions per method.** Every `*_rate` divides by `n_th_turns`, and the arms differ on it: at PTO's endpoint the K=5 arm takes ~2.4 *more* therapist turns per session (12.84 vs 10.41 at iteration 8), while at GRPO's the K=5 arm takes ~4.0 *fewer* (11.31 vs 15.34 at iteration 5). A rate can therefore move with the count held fixed, in either direction, so the raw per-session counts are tested as their own family (`MICI_COUNT_CHANNELS`) — the over-praise gap survives on counts (PTO 1.50 vs 0.47 per session at iteration 8), so *that* channel is real. ⚠ Turn count is not the only moving denominator: GRPO's K=5 turns are also ~1.7× longer, so neither per-turn nor per-session is denominator-free. The share-of-total columns in `k_mici_composition` are.
2. **Aggregate.** A channel closing is not the same as a behaviour reducing. `MICI_BehaviorTotal` is the sum of all six MI-inconsistent acts; if it is unchanged while one component collapses, the policy **substituted** rather than improved. Check it before writing the word "mitigation" anywhere.

**Sign convention is §4c's — `+ ⇒ K=0 higher` — and it is NOT a valence.** These are behaviour counts. A positive Δ on a `LOWER_IS_BETTER` channel means the K=0 policy does *more of a bad thing*; a positive Δ on `mean_turn_len` means only that it writes longer turns. The forest figure colours by valence so the two never get conflated.

⚠ **Holm scope.** Corrected across the channels **within each (method, iteration, family)** — the families are `behavior.BEHAVIOR_CHANNEL_FAMILIES`, and the `family` column records which one each row belongs to. Correcting across all ~40 channels at once would treat "over-praise per turn" and "conversation length" as one hypothesis set; they are not.

In [ ]:
from eda_analysis import behavior

if S.VIEW != eda_analysis.RQ_I_VIEW:
    print(f"[skip] RQ-i behaviour channels are owned by the {eda_analysis.RQ_I_VIEW} view — see "
          f"results/{eda_analysis.RQ_I_VIEW}/tables/7_stats/k_{{paired_channels,means_channels,"
          f"mici_composition}} + figures/7_stats/k_{{channel_forest,overpraise_trajectory,"
          f"cost_benefit,channel_grid}}.")
else:
    KA = eda_analysis.cross_k_arms(S)                    # both K arms; same filters as the view
    CH = behavior.channel_scores_long(KA)
    present = set(CH.questionnaire.unique())
    FAMILIES = {f: [c for c in cs if c in present]
                for f, cs in behavior.BEHAVIOR_CHANNEL_FAMILIES.items()}
    FAMILIES = {f: cs for f, cs in FAMILIES.items() if cs}
    print("channel frame:", CH.shape, "| arms:", sorted(CH.arm.unique()))
    for f, cs in FAMILIES.items():
        print(f"   family {f!r}: {len(cs)} channels")

    # ── the TEST: per family, so each Holm correction covers one hypothesis set ───
    frames = []
    for fam, chans in FAMILIES.items():
        for m in ["PTO", "GRPO"]:
            C = stats.paired_k_comparison(CH, m, metrics=chans)
            if not C.empty:
                frames.append(C.assign(family=fam))
    KPC = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    if KPC.empty:
        print("K0-vs-K5 behaviour channels not comparable yet for either method.")
    else:
        view = KPC[["method", "family", "iteration", "metric", "n",
                    "mean_delta", "dz", "p", "p_holm"]].round(4)
        print("=== K0 - K5 on the behaviour channels (+ => the K=0 policy does MORE of it) ===")
        display(view[view.metric.isin(["MICI_BehaviorTotal", "MICI_OverPraise",
                                       "MICI_AdviseNoPermission", "MICI_Severity"])])
        eda_analysis.save_table(view, "k_paired_channels", caption="RQ-i on the BEHAVIOUR channels: K0 - K5 within each method at matched iterations, persona-paired Wilcoxon + dz + Holm. + => the K=0 policy does MORE of the channel — NOT 'better': these are behaviour counts, so read valence off constants.LOWER_IS_BETTER (every MICI_* channel is higher = worse) and treat mean_turn_len / conv_len / n_th_turns as unvalenced. Holm scope: corrected across channels WITHIN each (method, iteration, family); the `family` column names the set. Read the per-SESSION families before the per-TURN ones: every rate divides by n_th_turns, which itself differs between the arms.")

    # ── the LEVELS the test is computed on ───────────────────────────────────────
    ALL_CH = [c for cs in FAMILIES.values() for c in cs]
    KMC = pd.concat([M for M in (stats.k_means_by_iter(CH, m, metrics=ALL_CH)
                                 for m in ["PTO", "GRPO"]) if not M.empty],
                    ignore_index=True).round(4)
    if not KMC.empty:
        eda_analysis.save_table(KMC, "k_means_channels", caption="RQ-i behaviour-channel LEVELS: K=0 vs K=5 arm means per method x channel x iteration with the unpaired delta (+ => K0 higher) and each side's n. The companion to k_paired_channels — read dz/p there, never off this table. Iterations the K=5 arm never reached keep mean_K5/delta NaN. Every per-turn value here also appears in mici_behavior_by_iter / miti_detail_by_iter / session_shape_by_iter under the same column name, by construction.")

    # ── THE AGGREGATE CHECK: is the MI-inconsistent total lower, or just recomposed? ──
    MICI_PER_CONV = behavior.mici_detail_per_conv(KA)
    if not MICI_PER_CONV.empty:
        comp_cols = ["MICI_BehaviorTotal"] + list(behavior._MICI_RATE_BEHAVIORS) + ["MICI_Severity"]
        COMP = (MICI_PER_CONV.groupby(["arm", "method", "K", "iteration"], observed=True)
                [[c for c in comp_cols if c in MICI_PER_CONV.columns]].mean().reset_index())
        # Share of the session's MI-inconsistent acts each behaviour accounts for — the claim.
        for b in behavior._MICI_RATE_BEHAVIORS:
            if b in COMP.columns:
                COMP[f"{b}_share"] = COMP[b] / COMP["MICI_BehaviorTotal"].where(
                    COMP["MICI_BehaviorTotal"] > 0)
        display(COMP.round(4))
        eda_analysis.save_table(COMP.round(4), "k_mici_composition", caption="MI-inconsistent behaviour COMPOSITION per (arm, iteration): the per-session count of each of the 6 behaviours, their sum (MICI_BehaviorTotal), the severity global, and each behaviour's SHARE of the session total. The aggregate control behind every per-turn rate claim: a component collapsing while the total holds means the policy SUBSTITUTED one violation for another, not that it committed fewer. Counts are per session and so carry no n_th_turns denominator.")

    # ── FIGURES ──────────────────────────────────────────────────────────────────
    # PRIMARY = the per-SESSION count, because that is the quantity with no moving denominator
    # (the arms differ in therapist-turn count). The per-turn rate is kept as the secondary
    # trajectory only. Mixing the two in one dz frame would put a rate beside its own aggregate.
    HACK = "MICI_OverPraise"
    HACK_RATE = "MICI_OverPraise_rate"
    AGG = "MICI_BehaviorTotal"
    K_ARMS = [a for a in ["PTO_LA0", "PTO_LA5", "GRPO_LA0", "GRPO_LA5"]
              if a in set(CH.arm.unique())]

    # The onset is READ OFF THE TEST, never eyeballed: first iteration at which the paired
    # contrast on the hack channel clears Holm within its own family.
    onset = None
    if not KPC.empty:
        sig = KPC[(KPC.method == "PTO") & (KPC.metric == HACK) & (KPC.p_holm < 0.05)]
        onset = int(sig.iteration.min()) if not sig.empty else None
    print("PTO divergence onset on", HACK, "->", onset)

    CHW = behavior.channels_per_conv(KA)     # per-conv wide frame; the trajectory figs average it
    fig = eda_analysis.plotting.k_channel_trajectory_grid(
        CHW, [HACK, "MICI_AdviseNoPermission", AGG, "MICI_Severity"],
        arms=K_ARMS, onset={HACK: onset} if onset else None,
        suptitle="Look-ahead recomposes MI-inconsistency, it does not reduce it "
                 "(K=0 solid · K=5 dashed)")
    eda_analysis.save_fig(fig, "k_mici_composition_grid", caption="The substitution, in per-SESSION counts so no denominator is involved. Over-praise (top left) collapses under K=5 while advice-without-permission (top right) rises to take its place; the TOTAL number of MI-inconsistent acts (bottom left) is indistinguishable at the matched endpoint, and the severity global (bottom right) is WORSE under K=5. Shading marks the measured divergence onset on over-praise. Arm means over 96 personas; K=5 lines stop at their last scored iteration.")
    plt.show()

    fig = eda_analysis.plotting.k_channel_trajectory(
        CHW, HACK_RATE, arms=K_ARMS, annotate_from=onset,
        title="The over-praise channel, per policy iteration")
    eda_analysis.save_fig(fig, "k_overpraise_trajectory", caption="Over-praise per therapist turn, every arm, per iteration (K=0 solid, K=5 dashed). The single channel that carries the whole MICI rise in both K=0 arms; the K=5 arm is flat across every iteration it reached. Shading marks the measured divergence onset. Higher = worse. Read with k_mici_composition_grid: this channel closing did NOT lower the MI-inconsistent total.")
    plt.show()

    if not KPC.empty:
        last = int(KPC[KPC.method == "PTO"].iteration.max())
        fig = eda_analysis.plotting.k_channel_forest(
            KPC, iteration=last, method="PTO",
            caption="Sign convention + = the K=0 policy does MORE of the channel. Colour is "
                    "VALENCE, not direction: red = a MI-inconsistent behaviour K=0 does more of, "
                    "green = one K=5 does more of, grey = unvalenced session shape. Red AND green "
                    "bars both being large is the finding: the violations were swapped, not "
                    "removed. Hollow = did not clear Holm within its (iteration, family) set.")
        eda_analysis.save_fig(fig, "k_channel_forest", caption="Every behaviour channel's persona-paired dz at the last matched PTO iteration — the trade-off in one frame. + => the K=0 policy does more of it. Colour encodes valence (red = MI-inconsistent and higher under K=0; green = MI-inconsistent and higher under K=5; grey = unvalenced session shape); hollow = did not clear Holm within its (iteration, family) set. The figure's point is that red and green bars are BOTH large: look-ahead moved the policy along a different axis rather than up a quality axis.")
        plt.show()

        # Reward vs hack in one frame — needs BOTH contrasts, so re-run §4c's on this KS.
        KS_ = eda_analysis.cross_k_scores(S)
        RUB = pd.concat([C for C in (stats.paired_k_comparison(KS_, m) for m in ["PTO", "GRPO"])
                         if not C.empty], ignore_index=True)
        fig = eda_analysis.plotting.k_cost_benefit(
            pd.concat([RUB, KPC], ignore_index=True),
            reward_metric=cfg.focus_metric, hack_channel=HACK, method="PTO",
            control_channel=AGG)
        eda_analysis.save_fig(fig, "k_cost_benefit", caption="The RQ-i answer in one frame: the paired K contrast on the REWARD the run optimised, on the CHANNEL it was hacked through, and on the AGGREGATE that channel belongs to — all as dz, all + => K=0 higher. The reward line stays inside 'small' and the aggregate line hugs zero while the channel line climbs past 'large': the two arms are near-equivalent both by the objective AND by total MI-inconsistency, and far apart only in WHICH violation they commit. Iteration 0 is the two independently generated base models and is the noise floor. Circled markers cleared Holm within their own family.")
        plt.show()

### 4e · The COMPUTE axis — is the lever worth what it costs?  `[EVAL]`
**Purpose.** §4c/§4d ask the K question at matched **iteration**. This asks it at matched **budget**, because an iteration is not a fixed unit of spend: a K=5 GRPO step costs ~1.9× a K=0 step, and a whole PTO iteration costs a fraction of a GRPO one. Nothing else in this EDA carries a cost column, so every previous contrast has been implicitly comparing unequal spends.

**Where the hours come from.** `compute.iteration_compute` reconstructs them from artifact mtimes — **not** from `iteration_metadata.json`, whose `training_time_s` / `pref_pair_time_s` are per-PROCESS and so record only the last session of any resumed iteration (GRPO_LA5 iteration 1 logs 14,501 s for work spanning 7.7 h; PTO logs `pref_pair_time_s = 3.2 s` for a ~30 min build it reloaded from `pairs.csv`). An iteration is billed as `generate + build + train`, where `build` is PTO's preference-tree phase — its dominant cost, and the reason GRPO's per-step timing alone would not make the two methods comparable.

**What it changes.** The two GRPO arms turn out to be **budget-matched to within ~3%** despite one running twice as many iterations — so "GRPO_LA5 only reached iteration 5" is a statement about iteration count, not spend, and the matched-iteration tables above hand K=5 roughly 2× the compute per cell. At the iso-compute endpoint the K contrast on `MICI` **reverses sign** relative to the matched-iteration reading.

⚠ **Pairing.** Iso-compute compares *different* iterations across arms, so `file_index` pairing is invalid here — the 96 personas are reshuffled `seed + k + 1` every iteration. Everything in `compute` pairs on `persona_id`; the self-check pins it (`compute axis (GPU-hours)`).

⚠ **`budget_sweep` is the honest form of the question.** A fixed-endpoint iso-compute contrast freezes each arm at one iteration; the sweep instead lets *both* arms spend each budget as well as they can (best checkpoint reachable within it). Read the sweep before quoting any single iso-compute row — the lever's sign is a **function of budget**, not a constant.

**Sign convention:** `+ mean_delta ⇒ arm_a higher` (so on a `LOWER_IS_BETTER` metric a positive Δ means arm_a is *worse*) — the same convention as `stats.py`.

In [ ]:
from eda_analysis import compute

if S.VIEW != eda_analysis.RQ_I_VIEW:
    print(f"[skip] the compute axis is owned by the {eda_analysis.RQ_I_VIEW} view — see "
          f"results/{eda_analysis.RQ_I_VIEW}/tables/7_stats/"
          f"{{compute_by_arm,compute_by_iteration,iso_compute_contrast,budget_sweep}} + "
          f"figures/7_stats/{{compute_trajectory,budget_sweep,cost_breakdown}}.")
else:
    KA = eda_analysis.cross_k_arms(S)                    # both K arms; same filters as the view
    KS = eda_analysis.cross_k_scores(S)
    COMP = compute.iteration_compute(KA)
    if COMP.empty:
        print("no run artifacts readable — compute axis skipped (Drive symlinks offline?).")
    else:
        # ── 1 · what each arm actually cost ──────────────────────────────────────
        CSUM = compute.compute_summary(COMP).round(3)
        print("=== GPU-hours per arm (generate + build + train) ==="); display(CSUM)
        eda_analysis.save_table(CSUM, "compute_by_arm", caption="Cost per arm: iterations trained and GPU-hours split by phase (generate rollouts / build preference trees / train), reconstructed from artifact mtimes because iteration_metadata.json's timings are per-PROCESS and undercount every resumed iteration. `train_source` records which artifact timed the optimizer loop: GRPO writes one completions parquet per step, DPO writes none and is timed from TensorBoard wall_time. THE table that makes 'arm X only reached iteration N' checkable against what arm X cost. n_imputed counts intervals replaced by the phase median because they were a resume gap or a re-synced Drive mtime.")

        CBI = COMP[COMP.iteration > 0].round(4)
        eda_analysis.save_table(CBI, "compute_by_iteration", caption="Per (arm, iteration) cost: step count, median step seconds, the three phase durations, the iteration total and the running cum_gpu_h that every iso-compute contrast indexes on. cum_gpu_h at iteration k is the cost of having produced the iter-k adapter — i.e. the policy the score lake calls <Arm>_I{k} — so it joins directly onto scores_long.")

        MULT = compute.step_multiplier(COMP, "GRPO").round(3)
        if not MULT.empty:
            print("=== per-step cost of look-ahead (GRPO) ==="); display(MULT)
            eda_analysis.save_table(MULT, "k_step_multiplier", caption="What one optimizer step costs at K=5 vs K=0, per iteration. Reported per iteration and NOT pooled: iteration 1 ran at a different LOOKAHEAD_SUB_BATCH_SIZE and carried a much fatter API-latency tail, so its ratio is not the intrinsic price of look-ahead. The later iterations are.")

        # ── 2 · the contrasts at matched BUDGET ──────────────────────────────────
        ISO_PAIRS = [("GRPO_LA5", "GRPO_LA0"), ("PTO_LA5", "PTO_LA0"),
                     ("PTO_LA0", "GRPO_LA0"), ("PTO_LA5", "GRPO_LA5")]
        frames = []
        for a, b in ISO_PAIRS:
            if not {a, b} <= set(KS.arm.unique()):
                continue
            T = compute.iso_compute_contrast(KS, COMP, a, b)
            if not T.empty:
                frames.append(T)
        if frames:
            ISO = pd.concat(frames, ignore_index=True)
            view = ISO[["arm_a", "arm_b", "iter_a", "iter_b", "cum_gpu_h_a", "cum_gpu_h_b",
                        "budget_ratio", "metric", "n", "mean_delta", "dz", "p", "p_holm"]].round(4)
            print("=== matched-BUDGET contrasts (+ => arm_a higher) — endpoint rows ===")
            display(view[view.metric.isin(["Q1Q2", "MICI", "MITI"])
                         & (view.groupby(["arm_a", "arm_b"]).iter_a.transform("max") == view.iter_a)])
            eda_analysis.save_table(view, "iso_compute_contrast", caption="Every arm pair contrasted at MATCHED CUMULATIVE GPU-HOURS rather than matched iteration, persona-paired (iso-compute reads a DIFFERENT iteration from each arm, so file_index pairing would join unrelated conversations - personas reshuffle seed+k+1 every iteration). budget_ratio = arm_b's spend / arm_a's; anything outside ~0.9-1.1 is not an iso-compute comparison and should not be quoted as one. + mean_delta => arm_a higher, so on MICI (lower=better) a positive delta means arm_a is WORSE. Holm scope: across rubrics within each budget-matched pair.")

        # ── 3 · the sign as a FUNCTION of budget, not at one point ───────────────
        sweeps = []
        for a, b in ISO_PAIRS:
            if not {a, b} <= set(KS.arm.unique()):
                continue
            W = compute.budget_sweep(KS, COMP, a, b, metric=cfg.focus_metric)
            if not W.empty:
                sweeps.append(W.assign(arm_a=a, arm_b=b))
        if sweeps:
            SW = pd.concat(sweeps, ignore_index=True).round(4)
            print(f"=== budget sweep on {cfg.focus_metric} (best checkpoint within budget, each arm) ===")
            display(SW)
            eda_analysis.save_table(SW, "budget_sweep", caption="Was the lever worth it, as a function of SPEND? At each budget both arms are represented by the best checkpoint they could have reached for that money (best on the focus metric under the active judge), persona-paired. The honest form of the iso-compute question: a fixed-endpoint contrast freezes each arm at one iteration, this lets each spend its budget as well as it can. The sign is NOT constant across budgets - quote the curve, not a point.")

        # ── FIGURES ──────────────────────────────────────────────────────────────
        BYC = compute.score_by_compute(KS, COMP, metric=cfg.focus_metric)
        fig = eda_analysis.plotting.compute_trajectory(BYC, metric=cfg.focus_metric)
        eda_analysis.save_fig(fig, "compute_trajectory", caption="THE figure this EDA was missing: the primary metric against cumulative GPU-hours instead of iteration, mean +/- SEM over the 96 personas, with iteration numbers annotated on the markers so the two axes can be read against each other. Unequal marker spacing along x IS the finding - an arm whose iterations are cheap gets many markers close together, and two arms that 'stopped at different iterations' can end at the same x.")
        plt.show()

        fig = eda_analysis.plotting.cost_breakdown(CSUM)
        eda_analysis.save_fig(fig, "cost_breakdown", caption="Where each arm's GPU-hours go, split into generate / build / train. Explains the cost gap rather than asserting it: PTO's dominant phase is the preference-tree build (absent in GRPO, whose reward computation happens inside the training loop instead), which is why per-step timings alone cannot compare the two methods.")
        plt.show()

        if sweeps:
            main = SW[(SW.arm_a == "GRPO_LA5") & (SW.arm_b == "GRPO_LA0")]
            if not main.empty:
                fig = eda_analysis.plotting.budget_sweep_plot(
                    main, label_a="GRPO K=5", label_b="GRPO K=0")
                eda_analysis.save_fig(fig, "budget_sweep", caption="The look-ahead lever's paired effect size as a function of budget, each arm represented by its best checkpoint within that budget. Filled = cleared p<.05. The curve crosses zero: at small budgets look-ahead is clearly WORSE (it buys fewer iterations for the money), and only at the largest budgets measured does it draw level or ahead. A single iso-compute row cannot express this, which is why the sweep is the artifact to quote.")
                plt.show()

## 5 · Climb rate, rankings, factor structure  `[EVAL]`
**Purpose.** Q1+Q2 OLS slope + Spearman ρ per arm (climb rate vs endpoint); and the rubric PCA (PC1 share → do the rubrics collapse to ~one latent factor?). **Note:** `slope_by_arm` reports ρ/slope but no p — the pooled persona×iteration rows are not independent, so trajectory p-values are descriptive; the repeated-measures-correct omnibus is the Friedman table in §2. **PCA caveats:** the PC1 91%→~55% drop when the further metrics are added is *partly mechanical* (appending less-correlated columns necessarily lowers the top eigenvalue's share) — read it as "a second dimension exists," not a calibrated effect size; and note the PCA is **pooled** over all convs/iters/arms. **Bootstrap CIs** everywhere (§1, §3) use a fixed seed (12345) → reproducible but not a source of independent uncertainty beyond the resample.

In [ ]:
SL = stats.filter_thin_arms(pd.DataFrame([stats.trajectory_test(S.SCORES, a, m)
      for a in sorted(S.SCORES.arm.unique()) for m in S.METRICS]), S.SCORES)
SLv = SL[["arm", "metric", "spearman_rho", "ols_slope", "peak_iter", "final_iter"]].round(4)
display(SLv)
eda_analysis.save_table(SLv, "slope_by_arm", caption="Per-iteration OLS slope + Spearman rho per arm x metric (climb rate; peak vs final iteration flags a regression). Thin arms dropped.")
PCA = pd.DataFrame([{"arm": a, "PC1_pct": round(100*stats.rubric_pca(S.SCORES[S.SCORES.arm==a])["explained_variance_ratio"][0], 1)}
                    for a in sorted(S.SCORES.arm.unique()) if stats.rubric_pca(S.SCORES[S.SCORES.arm==a])["explained_variance_ratio"]])
display(PCA)
eda_analysis.save_table(PCA, "rubric_pca_pc1", caption="Variance explained by PC1 of the rubric scores per arm (dominant PC1 => rubrics ~ one latent factor).")

## 6 · GRPO iter-9 anomaly check  `[EVAL]`
**Purpose.** The all-metric trajectory grid shows GRPO_LA0 dipping at **iter 9 across almost every metric simultaneously**, then partially recovering at 10 (while Q1+Q2 declines at both 9 and 10). Persona-paired deltas it8→9, it9→10, it8→10 test whether iter 9 is a one-iteration dip (eval-noise / transient policy state) vs the start of the regression. **Read:** significant negative it9−it8 followed by positive it10−it9 = a transient dip; monotonic negative it8→10 on Q1+Q2 = the real reward-hack regression.

In [ ]:
ARM = "GRPO_LA0"
a = S.SCORES[S.SCORES.arm == ARM]
need = {8, 9, 10}
if not a.empty and need <= set(a.iteration.unique()):
    model_at = lambda it: a[a.iteration == it].model.iloc[0]
    mets = [m for m in ["Q1Q2", "MITI", "WAI-SR"] if m in S.METRICS]
    CHK = pd.concat([stats.compare_two_models(a, model_at(hi), model_at(lo), mets).assign(contrast=f"it{hi}-it{lo}")
                     for lo, hi in [(8, 9), (9, 10), (8, 10)]], ignore_index=True)
    view = CHK[["contrast", "metric", "n", "mean_delta", "dz", "p", "p_holm"]].round(4)
    display(view)
    eda_analysis.save_table(view, "grpo_iter9_check",
        caption="GRPO_LA0 iter-9 anomaly: persona-paired deltas it8->9, it9->10, it8->10 on Q1+Q2/MITI/WAI-SR (contrast label hi-lo, so +=later higher). A significant negative it9-it8 followed by a positive it10-it9 = a one-iteration dip (eval-noise or transient policy state), distinct from the monotonic Q1+Q2 reward-hack regression. Holm scope: p_holm is corrected across the 3 metrics WITHIN each contrast.")
else:
    print(f"{ARM} iters 8-10 not all scored in this view — anomaly check skipped.")

## 7 · Artifact index
Refresh the per-view `results/<VIEW>/INDEX.md` (7_Stats runs last, so this call captures every family).

In [ ]:
print("index ->", eda_analysis.build_index())